[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/en_clase/S12_02_redes_recurrentes_en_clase.ipynb)

# S12 · 02 — Redes recurrentes (RNN, LSTM, GRU) para predecir la ocupación · versión en clase

⏱ 45 min en clase · Sprint 12 · UD 4.7

**Contexto TurisData.** Mismo problema que en `S12_01` (previsión de ocupación a 7 días), ahora con redes que "leen" la secuencia día a día y tienen memoria.

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. El mini-reto de la ventana corta y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2 y 3 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, random, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar la ocupación diaria

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántos días de datos hay?
2. ¿Qué diferencia hay entre una red normal y una recurrente?
3. ¿En qué se diferencian LSTM y GRU de SimpleRNN?

In [ ]:
ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")
print("TensorFlow", tf.__version__, "| filas:", len(ocup))

### 👀 A3 · EJEMPLO — Crear las variables de entrada y reservar la prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Cuántas variables describen cada día?
2. ¿Por qué se usan seno y coseno para el día de la semana?
3. ¿Cuántos días hay en entrenamiento y en prueba?

In [ ]:
DIAS_TEST, VENTANA, HORIZONTE = 91, 28, 7

df = pd.DataFrame(index=ocup.index)
df["ocupacion"] = ocup["ocupacion_pct"]
df["festivo"] = ocup["festivo"]
dsem, danio = ocup.index.dayofweek.values, ocup.index.dayofyear.values
df["sem_sin"], df["sem_cos"] = np.sin(2 * np.pi * dsem / 7), np.cos(2 * np.pi * dsem / 7)
df["anio_sin"], df["anio_cos"] = np.sin(2 * np.pi * danio / 365.25), np.cos(2 * np.pi * danio / 365.25)

n_ent = len(df) - DIAS_TEST
print("Días de entrenamiento:", n_ent, "| días de prueba:", DIAS_TEST)

### ✏️ A4 · EJERCICIO 1 de 3 — Escalar sin fuga de datos

**Tu turno:** Crea `escalador = MinMaxScaler()`, ajústalo con la columna de ocupación de las primeras `n_ent` filas y guarda en `ocup_esc` la ocupación de **toda** la serie transformada (array de una dimensión).

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Por qué el escalador se ajusta solo con el entrenamiento?
2. ¿Qué hace `fit` y qué hace `transform`?
3. ¿Qué puede pasar con los valores escalados de la prueba?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: fit sobre df[["ocupacion"]].iloc[:n_ent]; transform sobre toda la columna; usa .ravel()
escalador = ...
ocup_esc = ...

assert len(ocup_esc) == len(df), "ocup_esc debe tener una posición por día (transforma toda la serie)."
assert abs(ocup_esc[:n_ent].min()) < 1e-9 and abs(ocup_esc[:n_ent].max() - 1) < 1e-9, \
    "El escalador debe ajustarse solo con el entrenamiento: en entrenamiento el mínimo es 0 y el máximo 1."
print("✔ Ejercicio 1 correcto")

### 👀 A5 · EJEMPLO — Construir la matriz de variables escaladas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué forma tiene la matriz de variables?
2. ¿Qué es `objetivo`?
3. ¿Por qué se pasa a `float32`?

In [ ]:
X_feat = df.copy()
X_feat["ocupacion"] = ocup_esc
matriz = X_feat.values.astype("float32")          # (días, 6 variables)
objetivo = ocup_esc.astype("float32")             # lo que queremos predecir
print("Matriz de variables:", matriz.shape)

### ✏️ A6 · EJERCICIO 2 de 3 — Crear las ventanas deslizantes

**Tu turno:** Completa `crear_ventanas(origenes)`, que recibe una lista de posiciones `t` y devuelve `X` con forma `(n, 28, 6)` e `y` con forma `(n, 7)`.

**Debe salir:** `Entrenamiento: (971, 28, 6) (971, 7) | Prueba: (13, 28, 6) (13, 7)` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Cuántos ejemplos de entrenamiento y de prueba salen?
2. ¿Qué forma tiene `X` y qué significa cada dimensión?
3. ¿Por qué los ejemplos de entrenamiento están tan solapados?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
def crear_ventanas(origenes):
    # pista: para cada t: matriz[t-VENTANA:t] y objetivo[t:t+HORIZONTE]; apila con np.array
    ...

# Orígenes de entrenamiento: todas las posiciones cuyo futuro (7 días) cae DENTRO del entrenamiento
orig_ent = list(range(VENTANA, n_ent - HORIZONTE + 1))
# Orígenes de prueba: igual que en S12_01, uno por semana
orig_test = [o for o in range(n_ent, len(df), HORIZONTE) if o + HORIZONTE <= len(df)]

X_ent, y_ent = crear_ventanas(orig_ent)
X_test, y_test = crear_ventanas(orig_test)
print("Entrenamiento:", X_ent.shape, y_ent.shape, "| Prueba:", X_test.shape, y_test.shape)
assert X_ent.shape[1:] == (VENTANA, 6) and y_ent.shape[1] == HORIZONTE, "X debe ser (n, 28, 6) e y (n, 7)."
assert np.allclose(X_ent[0, -1], matriz[VENTANA - 1]), "El último día de la ventana del origen t es t-1."
print("✔ Ejercicio 2 correcto")

### 👀 A7 · EJEMPLO — Comprobar que no hay fuga de datos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué comprueba el `assert` de esta celda?
2. ¿Cuál es el último día usado en entrenamiento?
3. ¿Por qué es importante esta comprobación?

In [ ]:
assert max(orig_ent) + HORIZONTE - 1 < n_ent, "Fuga: una ventana de entrenamiento llega a días de prueba."
print("Sin fuga: el último día usado en entrenamiento es", df.index[max(orig_ent) + HORIZONTE - 1].date(),
      "y la prueba empieza el", df.index[n_ent].date())

### 👀 A8 · EJEMPLO — El baseline estacional en la misma escala

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuánto vale el MAE del baseline?
2. ¿Por qué se multiplica por el rango del escalador?
3. ¿Qué hace el baseline estacional?

In [ ]:
def mae_puntos(y_esc, pred_esc):
    """Error absoluto medio en puntos de ocupación (deshaciendo el escalado)."""
    rango = escalador.data_max_[0] - escalador.data_min_[0]
    return float(np.mean(np.abs(y_esc - pred_esc)) * rango)

pred_base = np.array([objetivo[o - 7:o] for o in orig_test])       # última semana conocida
mae_base = mae_puntos(y_test, pred_base)
print(f"MAE baseline estacional: {mae_base:.2f} puntos")

### 👀 A9 · EJEMPLO — Definir construir y entrenar las redes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Qué cambia entre las tres redes?
2. ¿Qué hace `EarlyStopping(patience=6, restore_best_weights=True)`?
3. ¿Qué devuelve la capa `Dense(HORIZONTE)`?

In [ ]:
def construir(tipo):
    capa = {"SimpleRNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[tipo]
    modelo = keras.Sequential([
        layers.Input(shape=(VENTANA, 6)),
        capa(32),
        layers.Dense(HORIZONTE),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(0.005), loss="mae")
    return modelo

def entrenar(tipo, epocas=40):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(tipo)
    n_val = 60
    parada = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss")
    h = m.fit(X_ent[:-n_val], y_ent[:-n_val], validation_data=(X_ent[-n_val:], y_ent[-n_val:]),
              epochs=epocas, batch_size=32, shuffle=True, callbacks=[parada], verbose=0)
    return m, h

### ✏️ A10 · EJERCICIO 3 de 3 — Entrenar SimpleRNN, LSTM y GRU

**Tu turno:** Entrena las tres redes en un diccionario `modelos` con claves `"SimpleRNN"`, `"LSTM"` y `"GRU"`, donde cada valor es la tupla `(modelo, historial)` que devuelve `entrenar`. Después se calcula el MAE de cada una en `resultados`.

**Debe salir:** `SimpleRNN  MAE prueba: 3.79 puntos | épocas: 31`, `LSTM  3.51 | 15`, `GRU  3.28 | 22`, `Baseline  4.46` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué MAE obtiene cada red y cuál es la mejor?
2. ¿Ganan las redes al baseline?
3. ¿Qué indica que la LSTM haya parado a las 15 épocas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: recorre los tres nombres y llama a entrenar(nombre)
modelos = ...

assert set(modelos) == {"SimpleRNN", "LSTM", "GRU"}, "modelos debe tener las claves SimpleRNN, LSTM y GRU."

resultados = {}
for nombre, (m, h) in modelos.items():
    pred = m.predict(X_test, verbose=0)
    resultados[nombre] = mae_puntos(y_test, pred)
    print(f"{nombre:10s} MAE prueba: {resultados[nombre]:.2f} puntos | épocas: {len(h.history['loss'])}")
print(f"{'Baseline':10s} MAE prueba: {mae_base:.2f} puntos")
print("✔ Ejercicio 3 correcto")

### 👀 A11 · EJEMPLO — Ver las curvas de aprendizaje de las tres redes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué dos curvas hay en cada gráfico?
2. ¿Qué señal indica que conviene parar el entrenamiento?
3. ¿Por qué comparten el eje vertical?

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (nombre, (m, h)) in zip(ejes, modelos.items()):
    ax.plot(h.history["loss"], label="entrenamiento"); ax.plot(h.history["val_loss"], label="validación")
    ax.set_title(nombre); ax.set_xlabel("Época")
ejes[0].set_ylabel("Error (MAE, escala 0-1)"); ejes[0].legend()
plt.tight_layout(); plt.show()

### 👀 A12 · EJEMPLO — Comparar con el baseline y ver una previsión

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué red es la mejor y cuánto gana al baseline?
2. ¿Con qué más hay que comparar antes de poner una red en producción?
3. ¿Por qué `inv` deshace el escalado en el gráfico?

In [ ]:
mejor = min(resultados, key=resultados.get)
comp = pd.Series({**resultados, "Baseline estacional": mae_base}).sort_values()
print(comp.round(2).to_string())

m = modelos[mejor][0]
pred = m.predict(X_test, verbose=0)
rango = escalador.data_max_[0] - escalador.data_min_[0]
inv = lambda a: a * rango + escalador.data_min_[0]
fig, ax = plt.subplots()
real_cont = inv(y_test).ravel(); pred_cont = inv(pred).ravel(); base_cont = inv(pred_base).ravel()
ind = df.index[orig_test[0]:orig_test[0] + len(real_cont)]
ax.plot(ind, real_cont, color="black", label="Real"); ax.plot(ind, base_cont, alpha=.7, label="Baseline t-7")
ax.plot(ind, pred_cont, label=f"Red {mejor}")
ax.set_title("Previsión a 7 días (últimas 13 semanas)"); ax.set_ylabel("Ocupación (%)"); ax.legend()
plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_02_redes_recurrentes.ipynb)** trae el mini-reto de la ventana corta y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib`, `tensorflow`, `keras` (con sus capas) y `MinMaxScaler` de `scikit-learn`.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow; `keras.utils.set_random_seed(SEMILLA)` fija las semillas de Python, NumPy y TensorFlow para que los resultados se repitan; `plt.rcParams.update(...)` da un estilo común a los gráficos.
- **Qué debe salir:**
    - `Listo ✔`. Importar TensorFlow tarda unos segundos.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 1. Idea: una red con memoria

#### A2 · Ejemplo — Cargar la ocupación diaria

**Contexto.** Mismo problema que en `S12_01` (previsión de ocupación a 7 días), ahora con redes que «leen» la secuencia día a día y tienen memoria.

Una red normal mira cada fila por separado. Una **red recurrente** lee la secuencia paso a paso y va
actualizando un "resumen" de lo visto (su **estado**).
- **SimpleRNN**: resumen simple; tiende a olvidar lo lejano.
- **LSTM**: añade *puertas* que deciden qué recordar, qué olvidar y qué sacar.
- **GRU**: versión más ligera de la LSTM (menos puertas, más rápida).

Para entrenarlas, la serie se corta en **ventanas**: cada ejemplo es un trozo de 28 días (entrada) y los
7 días siguientes (salida).

- **Qué hace:**
    - `pd.read_csv(..., parse_dates=["fecha"]).set_index("fecha")` lee la ocupación diaria y usa la fecha como índice.
    - El `print` muestra la versión de TensorFlow y el número de filas.
- **Qué debe salir:**
    - `TensorFlow 2.21.0 | filas: 1096` (la versión puede ser otra).
- **Cómo leerlo:**
    - 1 096 días de ocupación: unos tres años de datos, pocos para una red neuronal (de ahí el cuidado con el sobreajuste).
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos días de datos hay?* → **1 096** (de 2023 a 2025).
2. *¿Qué diferencia hay entre una red normal y una recurrente?* → Una red normal mira cada fila **por separado**; una recurrente lee la secuencia **paso a paso** y mantiene un **estado** (memoria).
3. *¿En qué se diferencian LSTM y GRU de SimpleRNN?* → Tienen **puertas** que deciden qué recordar y qué olvidar (la GRU, con menos puertas, es más ligera); la SimpleRNN tiende a olvidar lo lejano.

</details>

## 2. Variables de entrada y partición temporal

#### A3 · Ejemplo — Crear las variables de entrada y reservar la prueba

Cada día será un vector con: ocupación, festivo y el día de la semana y del año en forma de seno/coseno
(así "domingo" y "lunes" quedan cerca). **Todo es información conocida** hasta ese día.

- **Qué hace:**
    - `DIAS_TEST, VENTANA, HORIZONTE = 91, 28, 7` fija el tamaño de la prueba (91 días), de la ventana de entrada (28 días) y del horizonte de predicción (7 días).
    - `df` reúne, para cada día, la ocupación, el festivo y el día de la semana y del año como seno y coseno (`sem_sin`, `sem_cos`, `anio_sin`, `anio_cos`), de modo que «domingo» y «lunes» quedan cerca.
    - `n_ent` es el número de días de entrenamiento (todos menos los últimos 91).
- **Qué debe salir:**
    - `Días de entrenamiento: 1005 | días de prueba: 91`.
- **Cómo leerlo:**
    - Seis variables por día (ocupación, festivo y cuatro de calendario), todas **conocidas** hasta ese día. Los últimos 91 días (13 semanas) quedan reservados para la prueba.
- **Si falla:**
    - `NameError: name 'ocup' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas variables describen cada día?* → **6**: ocupación, festivo y cuatro de calendario (seno y coseno de la semana y del año).
2. *¿Por qué se usan seno y coseno para el día de la semana?* → Para que **domingo y lunes queden cerca** (el calendario es circular).
3. *¿Cuántos días hay en entrenamiento y en prueba?* → **1 005** y **91**.

</details>

## 3. Escalado SIN fuga

#### A4 · Ejercicio 1 — Escalar sin fuga de datos

Las redes aprenden mal con números grandes; se escala la ocupación a [0, 1].
**Regla de oro:** el escalador se ajusta (`fit`) **solo con el entrenamiento**; con la prueba solo se aplica (`transform`).
Si usáramos el máximo y el mínimo de toda la serie, estaríamos filtrando información del futuro.

- **Qué hace:**
    - `MinMaxScaler()` lleva los valores al rango [0, 1]. `escalador.fit(df[["ocupacion"]].iloc[:n_ent])` aprende el mínimo y el máximo **solo del entrenamiento**.
    - `escalador.transform(df[["ocupacion"]]).ravel()` aplica ese escalado a **toda** la serie y la deja en una dimensión (`ocup_esc`).
    - Las comprobaciones verifican que hay un valor por día y que en el entrenamiento el mínimo es 0 y el máximo 1.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - Como el escalador solo conoce el entrenamiento, los valores de la prueba pueden salir **fuera de [0, 1]** (si superan el máximo o el mínimo del pasado): es lo correcto, porque en producción tampoco se conoce el futuro.
- **Si falla:**
    - `ocup_esc debe tener una posición por día` → has transformado solo el entrenamiento: hay que transformar toda la serie.
    - `El escalador debe ajustarse solo con el entrenamiento ...` → has hecho `fit` con toda la serie.
    - `NameError: name 'n_ent' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué el escalador se ajusta solo con el entrenamiento?* → Porque, si no, el máximo y el mínimo de toda la serie **filtrarían información del futuro** (fuga de datos).
2. *¿Qué hace `fit` y qué hace `transform`?* → `fit` **aprende** el mínimo y el máximo; `transform` **aplica** el escalado.
3. *¿Qué puede pasar con los valores escalados de la prueba?* → Que salgan **fuera de [0, 1]**, y es correcto: el escalador solo conoce el pasado.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
escalador = MinMaxScaler()
escalador.fit(df[["ocupacion"]].iloc[:n_ent])
ocup_esc = escalador.transform(df[["ocupacion"]]).ravel()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A5 · Ejemplo — Construir la matriz de variables escaladas

- **Qué hace:**
    - `X_feat` es la tabla de variables con la ocupación sustituida por su versión escalada (`ocup_esc`).
    - `matriz` es esa tabla como array de `float32` (días × 6 variables) y `objetivo` es la ocupación escalada, lo que queremos predecir.
- **Qué debe salir:**
    - `Matriz de variables: (1096, 6)`.
- **Cómo leerlo:**
    - Cada fila es un día y cada columna una variable; de esta matriz se cortarán las ventanas.
- **Si falla:**
    - `NameError: name 'ocup_esc' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene la matriz de variables?* → **(1096, 6)**: 1 096 días por 6 variables.
2. *¿Qué es `objetivo`?* → La **ocupación escalada**: lo que la red debe predecir.
3. *¿Por qué se pasa a `float32`?* → Porque es el tipo de número con el que **trabaja Keras** (y ocupa menos memoria).

</details>

## 4. Ventanas deslizantes

#### A6 · Ejercicio 2 — Crear las ventanas deslizantes

Para cada posición `t` (el "origen"): entrada = días `t-28 … t-1`; salida = días `t … t+6`.

- **Qué hace:**
    - `crear_ventanas(origenes)` recorre cada origen `t` y toma como entrada los 28 días anteriores (`matriz[t - VENTANA:t]`) y como salida los 7 días siguientes (`objetivo[t:t + HORIZONTE]`); `np.array` los apila en `X` (n, 28, 6) e `y` (n, 7).
    - `orig_ent` son los orígenes de entrenamiento: todas las posiciones cuyos 7 días de futuro caen **dentro** del entrenamiento. `orig_test` es uno por semana en las 13 semanas de prueba.
    - Se crean `X_ent, y_ent` y `X_test, y_test`, se imprimen sus formas y las comprobaciones verifican las dimensiones y que el último día de la ventana del origen `t` es `t-1`.
- **Qué debe salir:**
    - `Entrenamiento: (971, 28, 6) (971, 7) | Prueba: (13, 28, 6) (13, 7)` y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Hay 971 ejemplos de entrenamiento (muy solapados entre sí: cada ventana desplaza un solo día respecto a la anterior) y 13 de prueba, una por semana.
- **Si falla:**
    - `X debe ser (n, 28, 6) e y (n, 7)` → los cortes de `matriz` y `objetivo` no tienen el tamaño `VENTANA` y `HORIZONTE`.
    - `El último día de la ventana del origen t es t-1` → la ventana debe terminar en `t-1` (`matriz[t - VENTANA:t]`, sin incluir `t`).
    - `NameError: name 'matriz' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos ejemplos de entrenamiento y de prueba salen?* → **971** y **13**.
2. *¿Qué forma tiene `X` y qué significa cada dimensión?* → `(n, 28, 6)`: **n** ejemplos, **28** días de historia y **6** variables.
3. *¿Por qué los ejemplos de entrenamiento están tan solapados?* → Porque cada ventana se desplaza **un solo día** respecto a la anterior.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def crear_ventanas(origenes):
    X = np.array([matriz[t - VENTANA:t] for t in origenes], dtype="float32")
    y = np.array([objetivo[t:t + HORIZONTE] for t in origenes], dtype="float32")
    return X, y
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A7 · Ejemplo — Comprobar que no hay fuga de datos

**Comprobación sin fuga:** ninguna ventana de entrenamiento debe tocar el periodo de prueba.

- **Qué hace:**
    - El `assert` verifica que el último día que usa una ventana de entrenamiento (origen + 6) cae antes de `n_ent`.
    - El `print` muestra ese último día y el primer día de la prueba.
- **Qué debe salir:**
    - `Sin fuga: el último día usado en entrenamiento es 2025-10-01 y la prueba empieza el 2025-10-02`.
- **Cómo leerlo:**
    - Ninguna ventana de entrenamiento toca la prueba: el entrenamiento acaba el 1 de octubre y la prueba empieza el 2.
- **Si falla:**
    - `Fuga: una ventana de entrenamiento llega a días de prueba.` → `orig_ent` incluye orígenes cuyo futuro cae en la prueba.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué comprueba el `assert` de esta celda?* → Que **ninguna ventana de entrenamiento** llega a los días de la prueba.
2. *¿Cuál es el último día usado en entrenamiento?* → El **2025-10-01** (la prueba empieza el 2025-10-02).
3. *¿Por qué es importante esta comprobación?* → Porque una fuga haría que la evaluación sea **demasiado optimista**.

</details>

## 5. Baseline en la misma escala de evaluación

#### A8 · Ejemplo — El baseline estacional en la misma escala

El baseline estacional (repetir la última semana) sobre las mismas 13 semanas. Medimos el MAE en **puntos de ocupación**.

- **Qué hace:**
    - `mae_puntos(y_esc, pred_esc)` calcula el error absoluto medio y lo multiplica por el rango del escalador, para expresarlo en **puntos de ocupación**.
    - `pred_base` es el baseline estacional: para cada origen, repite la última semana conocida (`objetivo[o - 7:o]`). `mae_base` es su error en la prueba.
- **Qué debe salir:**
    - `MAE baseline estacional: 4.46 puntos`.
- **Cómo leerlo:**
    - Este es el listón: **4,46 puntos** de error medio. Una red solo merece la pena si lo mejora (es el mismo valor que en `S12_01`).
- **Si falla:**
    - `NameError: name 'escalador' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el MAE del baseline?* → **4,46** puntos de ocupación.
2. *¿Por qué se multiplica por el rango del escalador?* → Para **deshacer el escalado** y medir el error en puntos de ocupación, no en la escala 0–1.
3. *¿Qué hace el baseline estacional?* → **Repite la última semana** conocida como previsión de los 7 días siguientes.

</details>

## 6. Construir y entrenar las tres redes

#### A9 · Ejemplo — Definir construir y entrenar las redes

La misma estructura, cambiando solo la capa recurrente. Modelos pequeños (32 unidades) y pocas épocas para que sea rápido.
Usamos como validación las **últimas 60 ventanas del entrenamiento** (sin barajar) y *early stopping*: se para cuando deja de mejorar.

- **Qué hace:**
    - `construir(tipo)` crea un modelo con una capa de entrada (28 días × 6 variables), **una** capa recurrente de 32 unidades (`SimpleRNN`, `LSTM` o `GRU`, según `tipo`) y una capa `Dense(7)` que da los 7 días siguientes. Se compila con `adam` (tasa 0,005) y pérdida `mae`.
    - `entrenar(tipo, epocas)` fija la semilla, construye la red y la entrena con las últimas 60 ventanas del entrenamiento como validación (sin barajar esas) y `EarlyStopping` (para si la validación no mejora en 6 épocas y restaura los mejores pesos). Devuelve el modelo y su historial.
- **Qué debe salir:**
    - No imprime nada: solo define las dos funciones.
- **Cómo leerlo:**
    - Las tres redes comparten la misma estructura: cambia **solo** la capa recurrente, para que la comparación sea justa.
- **Si falla:**
    - `NameError: name 'VENTANA' is not defined` → falta el ejemplo de las variables de entrada.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué cambia entre las tres redes?* → **Solo la capa recurrente** (`SimpleRNN`, `LSTM` o `GRU`).
2. *¿Qué hace `EarlyStopping(patience=6, restore_best_weights=True)`?* → Para el entrenamiento si la validación **no mejora en 6 épocas** y devuelve los pesos de la mejor época.
3. *¿Qué devuelve la capa `Dense(HORIZONTE)`?* → **7 valores**: la previsión de los 7 días siguientes.

</details>

#### A10 · Ejercicio 3 — Entrenar SimpleRNN, LSTM y GRU

**Ejercicio 3 · 🏫 en clase.** Entrena las tres redes en un diccionario `modelos` con claves `"SimpleRNN"`, `"LSTM"` y `"GRU"`,
donde cada valor es la tupla `(modelo, historial)` que devuelve `entrenar`. Después se calcula el MAE de cada una en `resultados`.

- **Qué hace:**
    - `{nombre: entrenar(nombre) for nombre in [...]}` entrena las tres redes y guarda, para cada nombre, la tupla `(modelo, historial)`.
    - La comprobación calcula, para cada una, su MAE en la prueba (`resultados`) con `mae_puntos` y escribe también cuántas épocas entrenó (`len(h.history['loss'])`).
- **Qué debe salir:**
    - `SimpleRNN  MAE prueba: 3.79 puntos | épocas: 31`, `LSTM  MAE prueba: 3.51 puntos | épocas: 15`, `GRU  MAE prueba: 3.28 puntos | épocas: 22` y `Baseline  MAE prueba: 4.46 puntos`, más `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Las tres redes **ganan al baseline** (4,46): la mejor, la GRU, con 3,28 (≈ 26 % menos de error). La LSTM paró a las 15 épocas por `EarlyStopping`. Con un solo entrenamiento y 13 semanas de prueba, las diferencias entre las tres redes son pequeñas.
- **Si falla:**
    - `modelos debe tener las claves SimpleRNN, LSTM y GRU` → falta alguna de las tres claves.
    - `NameError: name 'entrenar' is not defined` → falta el ejemplo anterior. Tarda un minuto en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué MAE obtiene cada red y cuál es la mejor?* → SimpleRNN **3,79**, LSTM **3,51** y GRU **3,28**: la mejor es la **GRU**.
2. *¿Ganan las redes al baseline?* → **Sí**, las tres: el baseline tiene 4,46.
3. *¿Qué indica que la LSTM haya parado a las 15 épocas?* → Que la **validación dejó de mejorar** y `EarlyStopping` cortó el entrenamiento.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
modelos = {nombre: entrenar(nombre) for nombre in ["SimpleRNN", "LSTM", "GRU"]}

resultados = {}
for nombre, (m, h) in modelos.items():
    pred = m.predict(X_test, verbose=0)
    resultados[nombre] = mae_puntos(y_test, pred)
    print(f"{nombre:10s} MAE prueba: {resultados[nombre]:.2f} puntos | épocas: {len(h.history['loss'])}")
print(f"{'Baseline':10s} MAE prueba: {mae_base:.2f} puntos")
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A11 · Ejemplo — Ver las curvas de aprendizaje de las tres redes

- **Qué hace:**
    - Para cada red dibuja la pérdida (MAE en escala 0–1) de entrenamiento y de validación por época, en tres gráficos con el mismo eje vertical.
- **Qué debe salir:**
    - Tres gráficos (SimpleRNN, LSTM y GRU), cada uno con dos curvas.
- **Cómo leerlo:**
    - En cada una, la curva de entrenamiento baja y la de validación se estanca: es justo lo que vigila `EarlyStopping` para parar a tiempo.
- **Si falla:**
    - `NameError: name 'modelos' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos curvas hay en cada gráfico?* → La **pérdida de entrenamiento** y la **de validación** por época.
2. *¿Qué señal indica que conviene parar el entrenamiento?* → Que la curva de **validación deja de bajar** mientras la de entrenamiento sigue bajando.
3. *¿Por qué comparten el eje vertical?* → Para poder **comparar las tres redes** a simple vista.

</details>

## 7. Comparar con el baseline y mirar una previsión

#### A12 · Ejemplo — Comparar con el baseline y ver una previsión

La pregunta importante no es "¿cuál red gana?" sino **"¿gana alguna al baseline?"**.

- **Qué hace:**
    - `mejor` es el nombre de la red con menor MAE y `comp` la tabla ordenada con las tres redes y el baseline.
    - El gráfico dibuja la ocupación real de las 13 semanas de prueba (en negro), la previsión del baseline y la de la mejor red; `inv` deshace el escalado para ver porcentajes de ocupación.
- **Qué debe salir:**
    - La tabla: `GRU 3.28` · `LSTM 3.51` · `SimpleRNN 3.79` · `Baseline estacional 4.46`, y un gráfico con tres líneas (real, baseline y GRU).
- **Cómo leerlo:**
    - Hay que mirar si **alguna red gana al baseline**: aquí sí, las tres. Pero en `S12_01` el modelo SARIMAX, más sencillo y explicable, logró 3,18: **la mejor red (3,28) ni siquiera lo supera**.
- **Si falla:**
    - `NameError: name 'resultados' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué red es la mejor y cuánto gana al baseline?* → La **GRU**, con 3,28 frente a 4,46 (≈ 26 % menos de error).
2. *¿Con qué más hay que comparar antes de poner una red en producción?* → Con **modelos más sencillos**, como el SARIMAX de `S12_01` (3,18), que aquí iguala a la mejor red.
3. *¿Por qué `inv` deshace el escalado en el gráfico?* → Para dibujar la ocupación en **porcentaje real** y no en la escala 0–1.

</details>